# Predicting the outcome of tennis matches using regression with multiple predictors
## Authors: Samuel Alkire, Ryan Sun, Mithun Rameshkumar


In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from google.colab import drive

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from sklearn.metrics import RocCurveDisplay

## Section 1: Organizing the Data


Begin by taking in the csv file with our dataseta and creating a dataframe

In [2]:
# Path to your uploaded CSV in Colab
matches_path = "/content/atp_matches_2024.csv"  # adjust if different

df = pd.read_csv(matches_path)
print(df.shape)
df.head()


FileNotFoundError: [Errno 2] No such file or directory: '/content/atp_matches_2024.csv'

Select only the columns that are needed for our analysis and drop rows with missing data

In [ ]:
needed_cols = [
    # Serve *inputs* – safe, not tautological
    'w_1stIn', 'w_svpt', 'w_ace', 'w_df',
    'l_1stIn', 'l_svpt', 'l_ace', 'l_df',

    # Pre-match variables
    'winner_rank', 'loser_rank',
    'winner_age', 'loser_age',
    'winner_ht', 'loser_ht',
    # Include player name
    'winner_name', 'loser_name'
]

df = df[needed_cols]
df = df.dropna()

Convert data to percentages where needed in order for effective comparison across matches

In [ ]:
# We convert all statistics to percentages in order to account for the fact that matches can be very different lengths

# First serve percentage (SAFE)
df['w_fs_pct'] = df['w_1stIn'] / df['w_svpt']
df['l_fs_pct'] = df['l_1stIn'] / df['l_svpt']

# Ace percentages
df['w_ace_pct'] = df['w_ace'] / df['w_svpt']
df['l_ace_pct'] = df['l_ace'] / df['l_svpt']

# Double fault percentages
df['w_df_pct'] = df['w_df'] / df['w_svpt']
df['l_df_pct'] = df['l_df'] / df['l_svpt']

# Differences
df['fs_pct_diff'] = df['w_fs_pct'] - df['l_fs_pct']
df['aces_diff'] = df['w_ace_pct'] - df['l_ace_pct']
df['df_diff'] = df['w_df_pct'] - df['l_df_pct']

# Pre-match diffs
df['ranking_diff'] = df['loser_rank'] - df['winner_rank']
df['height_diff'] = df['winner_ht']  - df['loser_ht']
df['age_diff']   = df['winner_age'] - df['loser_age']



## Section 2: Structure Data for Effective Regression

Create copy of dataframe, set the value to 1 for the winner

In [ ]:
# Use a copy of the full data for regression analysis
base = df.copy()

# Binary target (winner perspective)
base['y'] = 1

# New feature sets
all_features = [
    'fs_pct_diff', 'aces_diff', 'df_diff',
    'ranking_diff', 'height_diff', 'age_diff',
    'w_ace_pct', 'l_ace_pct', 'w_df_pct', 'l_df_pct', 'w_fs_pct', 'l_fs_pct',
    'winner_name', 'loser_name'
]

print("All features:", all_features)
base[all_features + ['y']].head()

Here we add entries but flipped so that we account for the losers perspective. This is necessary since otherwise all data has the outcome y=1 for winning

In [ ]:
neg = base.copy()

diff_features = [
    'fs_pct_diff', 'aces_diff', 'df_diff',
    'ranking_diff', 'height_diff', 'age_diff'
]

# Flip perspective: now Player1 = loser, so invert only diffs for numerical features
neg[diff_features] = -neg[diff_features]
neg['y'] = 0

# Combine both perspectives
model_df = pd.concat([base, neg], ignore_index=True)

# Drop any remaining NaNs
model_df = model_df.dropna(subset=all_features + ['y']).reset_index(drop=True)

print("Final modeling dataset shape:", model_df.shape)
model_df[all_features + ['y']].head()

## Section 3: Creating the Training and Test Data

To begin, we simply split the data randomly into 80% training and 20% test

In [ ]:
# Split into X (features) and y (target) from the augmented data
X = model_df[all_features]
y = model_df['y'].values

# Perform train-test split on the augmented data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

However, there is an issue with the test data that needs to be addressed. Currently, each row from the data contains data points from the match after it has already been played. This is an issue since we ultiamtely hope to predict the outcome of future matches...

In order to fix this, we must replace the match data from the test set with historical data from the training set. The general idea is as follows:



1.   Take any given match in the test data between Player A and Player B
2.   From the training data, calculate the average serve percentage, average ace percentage, and average double fault for Player A
3. Repeat step 2 for Player B
4. Subtract the percentages to get the difference and plug this into the test data

Note: we don't do this for rank, height, or age as these are all predictors that are available prior to the start of the match and thus can be used for future predictions



Here we define a function that calculates the average stats for each player based on the training data

In [ ]:
# This will create a player dictionary of their stats from the training data
player_pct = {}

def init_player():
    return {
        'fs_pct': 0,
        'ace_pct': 0,
        'df_pct': 0,
        'matches': 0
    }

for _, row in X_train.iterrows():

    winner = row['winner_name']
    loser  = row['loser_name']

    if winner not in player_pct:
        player_pct[winner] = init_player()
    if loser not in player_pct:
        player_pct[loser] = init_player()

    # Winner pct accumulation
    player_pct[winner]['fs_pct']  += row['w_fs_pct']
    player_pct[winner]['ace_pct'] += row['w_ace_pct']
    player_pct[winner]['df_pct']  += row['w_df_pct']
    player_pct[winner]['matches'] += 1

    # Loser pct accumulation
    player_pct[loser]['fs_pct']  += row['l_fs_pct']
    player_pct[loser]['ace_pct'] += row['l_ace_pct']
    player_pct[loser]['df_pct']  += row['l_df_pct']
    player_pct[loser]['matches'] += 1

# convert sums -> averages
for p, stats in player_pct.items():
    c = stats['matches']
    stats['fs_pct']  /= c
    stats['ace_pct'] /= c
    stats['df_pct']  /= c


This code then calculates the difference between the two players in each match found in the test data and replaces as needed

In [ ]:
# Player A vs Player B, pull average from training on both players, subsitute difference into test data
# Helper function: safely get averages, default to 0 if player not in dict
def get_player_avgs(name):
    if name in player_pct and player_pct[name]['matches'] > 0:
        return player_pct[name]
    else:
        return {'fs_pct': 0, 'ace_pct': 0, 'df_pct': 0}

# Create new columns for test feature differences
X_test['fs_pct_diff']  = np.nan
X_test['ace_pct_diff'] = np.nan
X_test['df_pct_diff']  = np.nan

for idx, row in X_test.iterrows():
    w = row['winner_name']
    l = row['loser_name']

    w_stats = get_player_avgs(w)
    l_stats = get_player_avgs(l)

    X_test.at[idx, 'fs_pct_diff']  = w_stats['fs_pct']  - l_stats['fs_pct']
    X_test.at[idx, 'ace_pct_diff'] = w_stats['ace_pct'] - l_stats['ace_pct']
    X_test.at[idx, 'df_pct_diff']  = w_stats['df_pct']  - l_stats['df_pct']

# Drop names and unneeded predictors after constructing differences
X_train = X_train.drop(['w_ace_pct', 'l_ace_pct', 'w_df_pct', 'l_df_pct', 'w_fs_pct', 'l_fs_pct',
    'winner_name', 'loser_name'], axis=1)
X_test  = X_test.drop(['w_ace_pct', 'l_ace_pct', 'w_df_pct', 'l_df_pct', 'w_fs_pct', 'l_fs_pct',
    'winner_name', 'loser_name'], axis=1)

print(X_test.head())

## Section 4: Scale Data and Calculate Regression Coefficients

In [ ]:
# Initialize and fit scaler on training numeric data
scaler = StandardScaler()
X_test = X_test[X_train.columns]
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test) # Transform test data using scaler fitted on train data

print("X_train_scaled shape:", X_train_scaled.shape)
print("X_test_scaled shape:", X_test_scaled.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

In [ ]:
log_reg = LogisticRegression(
    penalty='l2',
    C=1.0,
    solver='lbfgs',
    max_iter=1000
)

log_reg.fit(X_train_scaled, y_train)

y_pred = log_reg.predict(X_test_scaled)
y_proba = log_reg.predict_proba(X_test_scaled)[:, 1]

print("=== Plain Logistic Regression ===")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_proba))
print("\nClassification report:\n", classification_report(y_test, y_pred))

In [ ]:
coef_df = pd.DataFrame({
    'feature': diff_features,
    'coef': log_reg.coef_[0],
}).sort_values(by='coef', key=lambda s: s.abs(), ascending=False)

coef_df

## Section 5: Further Analysis

In [ ]:
coef_plot_df = coef_df.copy()
coef_plot_df['abs_coef'] = coef_plot_df['coef'].abs()
coef_plot_df = coef_plot_df.sort_values('abs_coef', ascending=True)

plt.figure(figsize=(8,6))
plt.barh(coef_plot_df['feature'], coef_plot_df['coef'])
plt.title("Feature Importance (Logistic Regression Coefficients)")
plt.xlabel("Coefficient Value (log-odds)")
plt.ylabel("Feature")
plt.axvline(0, color='gray', linestyle='--')
plt.show()

Heatmap that shows correlation between the predictors

In [ ]:
# Correlation matrix
corr = model_df[diff_features].corr()

plt.figure(figsize=(10, 8))
im = plt.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)

plt.colorbar(im, fraction=0.046, pad=0.04)
plt.xticks(
    ticks=np.arange(len(corr.columns)),
    labels=corr.columns,
    rotation=90
)
plt.yticks(
    ticks=np.arange(len(corr.index)),
    labels=corr.index
)
plt.title("Correlation Heatmap of Predictors")
plt.tight_layout()
plt.show()